# Deploy sparch LIF/SHD network to SpiNNaker

Trains a plain-LIF `sparch` model on Spiking Heidelberg Digits (SHD), exports its weights, and deploys the equivalent PyNN network as a batch job on real SpiNNaker hardware via the EBRAINS Neuromorphic Computing Platform. Follows the same batch-submission pattern as `SpiNNaker_01_test.ipynb` (via `nmpi`), not an interactive PyNN session.

**Prerequisites (done locally, before running this notebook):**
1. Train: `python run_exp.py --model_type LIF --dataset_name shd --data_folder shd_dataset --new_exp_folder exp/lif_shd_run1` (in the `sparch` repo)
2. Export: `python export_weights.py exp/lif_shd_run1/checkpoints/best_model.pth shd_lif_weights.npz`
3. Copy `shd_lif_weights.npz` to `hdc-language/src/data/`
4. Verify locally: `python verification/reference_lif_numpy.py` and `python verification/pynn_smoke_test.py --seed 42` (in the `hdc-language` repo) — see `verification/README.md` for the full tier breakdown.

This notebook only handles step 3 onward: generating the self-contained job script and submitting it to SpiNNaker.

In [ ]:
!pip install -U hbp_neuromorphic_platform
!pip install ebrains-drive

## Create our client

Because we are already logged into the HBP Collaboratory,
there is no need to enter your username or password.

If running this notebook locally on your own computer,
you will need to specify your username, and you will be
prompted for the password.

In [ ]:
import nmpi
import os
import time
import ebrains_drive
from ebrains_drive.client import DriveApiClient

client = nmpi.Client()

(This next cell is a workaround for a missing Collaboratory-2 functionality. Just run it and ignore the content...)

In [ ]:
class RepositoryInformation:
    def __init__(self,repository,nameInTheUrl):
        self.repository=repository
        self.nameInTheUrl=nameInTheUrl
    def toString(self):
        return "nameInTheUrl="+self.nameInTheUrl+", full name="+self.repository.name+", id="+self.repository.id

def findRepositoryInfoFromDriveDirectoryPath(homePath):
    name=homePath.replace("/mnt/user/shared/","")
    if name.find("/")>-1:
        name=name[:name.find("/")]
    thisCollabsTitle=name
    bearer_token = clb_oauth.get_token()
    ebrains_drive_client = ebrains_drive.connect(token=bearer_token)
    repo_by_title = ebrains_drive_client.repos.get_repos_by_filter("name",thisCollabsTitle)
    if len(repo_by_title)!=1:
        raise Exception("The repository for the collab name",thisCollabsTitle,"can not be found")
    owner=repo_by_title[0].owner
    collabNameInTheUrl=owner[:owner.rindex("-")]
    collabNameInTheUrl=collabNameInTheUrl[collabNameInTheUrl.find("-")+1:]
    return RepositoryInformation(repo_by_title[0],collabNameInTheUrl)

client = nmpi.Client()

dir=!pwd
repoInfo=findRepositoryInfoFromDriveDirectoryPath(dir[0])

checkForQuota=False
if checkForQuota:
    a=client.resource_requests(repoInfo.nameInTheUrl)
    anyAccepted=False
    if len(a)==0:
        print("This collab does not have any quota entry yet. Request a test quota.")
        client.create_resource_request(
        title="Test quota request for "+repoInfo.nameInTheUrl,
        collab_id=repoInfo.nameInTheUrl,
        abstract="Test quota request",
        submit=True)
    else:
        for entry in a:
            if entry["status"]=="accepted":
                print("An accepted quota request exists")
                anyAccepted=True
        if not anyAccepted:
            print("A quota request is present, but it has not yet been granted.")
    if not anyAccepted:
        raise Exception("This collab does not yet have an accepted quota entry.\nTherefore submitting jobs will not yet work.")

## Generate the self-contained job script

Unlike the reference notebook's toy demo (written inline via `%%file`), this job embeds real trained weights and real SHD test examples, so the generator script (`hdc-language/src/generate_spinnaker_script.py`) builds it for us: base64-encodes the exported `.npz` weights and a random sample of SHD test examples directly into the script text, and inlines `src/pynn_lif_model.py`'s network-building logic. This is necessary because the SpiNNaker job runner sandbox only sees the single uploaded file — it can't `import` from this repo or reliably read a second uploaded data file.

Run this from the `hdc-language` repo root (adjust paths if running this notebook from elsewhere on EBRAINS — e.g. after cloning the repo into your Collaboratory drive).

In [ ]:
import subprocess

REPO_ROOT = os.path.expanduser("~/hdc-language")  # adjust if cloned elsewhere
JOB_SCRIPT = os.path.expanduser("~/shd_lif_spinnaker.py")

subprocess.run([
    "python", os.path.join(REPO_ROOT, "src", "generate_spinnaker_script.py"),
    "--n_examples", "20",
    "--seed", "42",
    "--out", JOB_SCRIPT,
], check=True)

print(f"Generated {JOB_SCRIPT}")

## Submit the job to the Neuromorphic Computing Platform

and wait until it completes.

In [ ]:
print(nmpi.__version__)
print(client.job_server)
collab_id = repoInfo.nameInTheUrl
print("Using the repository ",collab_id," for quotas. Starting the job at",time.ctime())
job = client.submit_job(source=JOB_SCRIPT,
                        platform=nmpi.SPINNAKER,
                        collab_id=collab_id,
                        command="run.py",
                        wait=True)
print(job["log"])

## Download and inspect results

The job script prints per-example predictions and a summary accuracy line to stdout, captured in `job["log"]` above. This cell also downloads any files the job produced (there are no `.png` plots in this job, unlike the reference notebook's demo — the output is the printed accuracy summary itself).

In [ ]:
filenames = client.download_data(job, local_dir=os.path.expanduser("~"))
print("Fetched the files",filenames)

## Notes on interpreting results

- This job's predictions come from **spike-count argmax** (rate coding) on the readout population — a deliberate substitute for sparch's own softmax-accumulation readout, since no PyNN/SpiNNaker neuron primitive computes softmax across a population. See `src/pynn_lif_model.py`'s module docstring and `verification/README.md` for the full discussion of this and the other named approximation (reset semantics).
- Compare the printed accuracy against `verification/pynn_smoke_test.py`'s local `pyNN.nest` run on the **same `--seed`** — they should be close (both use the same network-building logic and the same scaling derivation); a large discrepancy would suggest a SpiNNaker-specific numerical or timing difference worth investigating (e.g. SpiNNaker's own internal timestep/precision constraints, which can differ from `pyNN.nest`'s).
- For actual hardware profiling (the point of this whole exercise), look at `job["log"]` and the EBRAINS job dashboard for timing/resource-usage metrics, not just the printed accuracy.